# Etapa 3 — Arquitectura de Dos Etapas (Candidate Generation + Ranking)
## Cuaderno de Reproducibilidad para Google Colab

Este cuaderno permite reproducir y verificar de forma liviana todo el pipeline de la **Etapa 3** de la tesis (Retrieval Two-Tower + Ranker MLP) sin duplicar código ni lógica, ejecutando directamente los módulos y scripts del proyecto.

> **NOTA VISIBLE METODOLÓGICA**:
> La tabla de referencia de este proyecto fue obtenida en la GPU local del autor (**NVIDIA GeForce RTX 4060 Laptop 8GB**, PyTorch `2.6.0+cu124`, CUDA 12.4, Windows 11). Las ejecuciones en Google Colab pueden presentar leves discrepancias numéricas en la tercera o cuarta decimal dependiendo del hardware asignado por Colab (ej. T4, L4, V100, A100), los controladores y la versión de PyTorch preinstalada. No se asume qué modelo de GPU asigne Google Colab.

--- 
## 1. Descarga del proyecto y posicionamiento en el directorio
Selecciona cómo cargar el código. Si tu repositorio es público en GitHub, la opción **`github`** es la más directa. Si aún no lo subiste a GitHub, podés arrastrar el archivo `.zip` del proyecto al panel de archivos de Colab (a la izquierda) y usar **`upload_zip`**.

In [ ]:
# @title Configuración de carga del proyecto
METODO = "github"  #@param ["github", "upload_zip", "drive_zip"]
REPO_URL = "https://github.com/TU_USUARIO/03-two-stage.git"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/03-two-stage.zip"  #@param {type:"string"}

import os, sys, glob

if METODO == "github":
    if "TU_USUARIO" in REPO_URL:
        raise ValueError("⚠️ Reemplazá 'TU_USUARIO' por tu usuario real de GitHub en la variable REPO_URL de arriba (o cambiá METODO a 'upload_zip').")
    repo_name = os.path.basename(REPO_URL).replace('.git', '')
    if not os.path.exists(f"/content/{repo_name}"):
        !git clone {REPO_URL} /content/{repo_name}
    %cd /content/{repo_name}

elif METODO == "upload_zip":
    # Busca cualquier zip subido a /content/
    zips = glob.glob("/content/*.zip")
    if not zips:
        raise FileNotFoundError("⚠️ Arrastrá el archivo .zip de tu proyecto al panel de archivos de Colab (a la izquierda) y volvé a correr esta celda.")
    zip_file = zips[0]
    print(f"Descomprimiendo {zip_file}...")
    !unzip -q -o {zip_file} -d /content/03-two-stage
    %cd /content/03-two-stage

elif METODO == "drive_zip":
    from google.colab import drive
    drive.mount('/content/drive')
    !unzip -q -o {DRIVE_ZIP_PATH} -d /content/03-two-stage
    %cd /content/03-two-stage

# Verificación automática: aseguramos que estamos parados en la raíz del proyecto
if not os.path.exists("requirements-colab.txt"):
    # Si se descomprimió con una subcarpeta adentro, nos movemos a ella
    subdirs = [d for d in glob.glob("**/requirements-colab.txt", recursive=True)]
    if subdirs:
        target_dir = os.path.dirname(subdirs[0])
        %cd {target_dir}

print("\nDirectorio de trabajo actual:", os.getcwd())
print("Archivos encontrados:", [f for f in os.listdir('.') if not f.startswith('.')])

--- 
## 2. Instalación de dependencias y verificación de GPU

In [ ]:
# Colab ya cuenta con PyTorch preinstalado con aceleración CUDA.
# Instalamos el resto de librerías necesarias con fallback seguro:
import os

if os.path.exists("requirements-colab.txt"):
    !pip install -q -r requirements-colab.txt
else:
    !pip install -q numpy pandas scikit-learn matplotlib pytest

print("Dependencias listas.")

In [ ]:
# Verificación del hardware asignado por Google Colab
import torch

!nvidia-smi

print("\n" + "=" * 50)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU : {torch.cuda.get_device_name(0)}")
    print(f"Versión de CUDA: {torch.version.cuda}")
    print(f"Capacidad CUDA  : {torch.cuda.get_device_capability(0)}")
else:
    print("ADVERTENCIA: No se detectó GPU activa. Activa la GPU en: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU.")
print("=" * 50)

--- 
## 3. Validación rápida del pipeline (`--quick`)
Ejecuta una corrida corta (1 época y subconjunto acotado de usuarios) para comprobar que la descarga del dataset MovieLens, la arquitectura Two-Tower, el Ranker MLP y el protocolo de evaluación funcionen de punta a punta en pocos segundos.

In [ ]:
!python scripts/run_stage3.py --quick --device auto

--- 
## 4. Corrida multi-semilla configurable
Ejecuta el pipeline completo de la Etapa 3 (30 épocas de Two-Tower y 8 épocas por ranker) a través de múltiples semillas para obtener estadísticas de estabilidad y contrastes pareados.

In [ ]:
# @title Parámetros de ejecución multi-semilla
# Puedes configurar la cantidad de semillas a ejecutar (por defecto 3: 42, 43, 44; o hasta 10: 42 a 51)

CANTIDAD_SEMILLAS = 3  #@param [3, 5, 10]
SEEDS_LIST = list(range(42, 42 + CANTIDAD_SEMILLAS))
seeds_arg = " ".join(map(str, SEEDS_LIST))

print(f"Ejecutando {len(SEEDS_LIST)} semillas: {SEEDS_LIST}")
!python scripts/run_multi_seed.py --seeds {seeds_arg} --device auto --suffix _colab

--- 
## 5. Visualización de resultados de la corrida en Colab

In [ ]:
from IPython.display import Image, display
import os

# Mostrar la curva de pérdida de entrenamiento del Retrieval
fig_path = "results/gpu/figures/loss_retrieval.png" if os.path.exists("results/gpu/figures/loss_retrieval.png") else "results/figures/loss_retrieval.png"
if os.path.exists(fig_path):
    print(f"Curva de pérdida guardada en: {fig_path}")
    display(Image(fig_path))
else:
    print("No se encontró la figura de pérdida.")

In [ ]:
import pandas as pd

dev_folder = "results/gpu" if os.path.exists("results/gpu/stage3_summary_colab.csv") else "results/cpu"
summary_colab = pd.read_csv(f"{dev_folder}/stage3_summary_colab.csv")
paired_colab = pd.read_csv(f"{dev_folder}/stage3_paired_differences_colab.csv")

print("=" * 80)
print(f"RESUMEN DE MÉTRICAS OBTENIDAS EN COLAB ({CANTIDAD_SEMILLAS} SEMILLAS)")
print("=" * 80)

piv_mean = summary_colab.pivot(index="modelo", columns="metrica", values="media")
piv_std = summary_colab.pivot(index="modelo", columns="metrica", values="desvio")

display_cols = ["precision_at_10", "recall_at_10", "ndcg_at_10", "cobertura_catalogo", "popularidad_media_recs"]
table = pd.DataFrame(index=piv_mean.index)
for c in display_cols:
    table[c] = piv_mean[c].map(lambda m: f"{m:.4f}") + " ± " + piv_std[c].map(lambda s: f"{s:.4f}")

order_pref = [
    "Popularidad (baseline)",
    "Retrieval (two-tower) solo",
    "Ranker[uniforme] catálogo completo (1 etapa)",
    "Two-Stage[uniforme] N=50",
    "Two-Stage[uniforme] N=200",
    "Ranker[mixto] catálogo completo (1 etapa)",
    "Two-Stage[mixto] N=50",
    "Two-Stage[mixto] N=200",
]
table["_rank"] = table.index.map(lambda x: order_pref.index(x) if x in order_pref else 99)
table = table.sort_values("_rank").drop(columns=["_rank"])
display(table)

print("\n" + "=" * 80)
print("DIFERENCIAS PAREADAS POR SEMILLA OBTENIDAS EN COLAB")
print("=" * 80)
display(paired_colab)

--- 
## 6. Comparación contra la referencia local (10 semillas en GPU RTX 4060)
A continuación se cargan los resultados de referencia registrados formalmente en la máquina del autor para contrastar la estabilidad de las conclusiones entre ambos entornos.

In [ ]:
ref_summary_path = "results/gpu/stage3_summary_10seeds.csv"
ref_paired_path = "results/gpu/stage3_paired_differences_10seeds.csv"

if os.path.exists(ref_summary_path):
    ref_summary = pd.read_csv(ref_summary_path)
    ref_paired = pd.read_csv(ref_paired_path)
    
    print("=" * 80)
    print("TABLA DE REFERENCIA LOCAL (10 SEMILLAS EN GPU RTX 4060, TORCH 2.6.0+cu124)")
    print("=" * 80)
    piv_ref_mean = ref_summary.pivot(index="modelo", columns="metrica", values="media")
    piv_ref_std = ref_summary.pivot(index="modelo", columns="metrica", values="desvio")
    
    ref_table = pd.DataFrame(index=piv_ref_mean.index)
    for c in display_cols:
        ref_table[c] = piv_ref_mean[c].map(lambda m: f"{m:.4f}") + " ± " + piv_ref_std[c].map(lambda s: f"{s:.4f}")
    ref_table["_rank"] = ref_table.index.map(lambda x: order_pref.index(x) if x in order_pref else 99)
    ref_table = ref_table.sort_values("_rank").drop(columns=["_rank"])
    display(ref_table)
    
    print("\n" + "=" * 80)
    print("DIFERENCIAS PAREADAS DE REFERENCIA LOCAL (10 SEMILLAS)")
    print("=" * 80)
    display(ref_paired)
else:
    print(f"Archivo de referencia {ref_summary_path} no encontrado en el directorio del proyecto.")